# 配套实践 05-01：分词、补齐与 mask

本练习对应基础篇第 05 章。我们构造一个教学用词级 tokenizer，观察 token id、padding、mask、embedding 和 masked pooling。真实项目应使用与预训练模型配套的 tokenizer。依赖：NumPy、PyTorch；CPU 即可运行。

<a href="https://qi-robotics.github.io/robot-world-model-tutorial/basics/05-language-encoding/" target="_blank">返回课程正文</a>

In [1]:
import torch  # 使用张量完成查表与带遮罩池化
torch.manual_seed(7)  # 固定随机种子以便复现实验结果

## 1. 建立教学词表

为了把重点放在数据接口上，下面的中文指令已经用空格标出词边界。词表只从训练指令建立，因此可以明确看到未知词怎样进入 `[UNK]`。

In [2]:
commands = ["拿起 红色 杯子", "把 蓝色 方块 放到 托盘", "推动 绿色 杯子 到 左侧"]  # 准备不同长度的机器人指令
special_tokens = ["[PAD]", "[UNK]", "[BOS]", "[EOS]"]  # 定义补齐、未知和序列边界符号
words = sorted({word for command in commands for word in command.split()})  # 收集训练指令出现过的词
vocab = {token: index for index, token in enumerate(special_tokens + words)}  # 为每个 token 分配唯一整数编号
inverse_vocab = {index: token for token, index in vocab.items()}  # 建立编号到 token 的反向映射便于检查
def encode(command):  # 定义把一条指令转换成 token id 的函数
    pieces = ["[BOS]"] + command.split() + ["[EOS]"]  # 在真实词前后加入序列边界
    return [vocab.get(piece, vocab["[UNK]"]) for piece in pieces]  # 未见词统一映射到未知 token
encoded = [encode(command) for command in commands]  # 编码全部示例指令
print("词表：", vocab)  # 显示 token 与 id 的对应关系
print("第一条指令 id：", encoded[0])  # 显示第一条序列的离散表示
assert vocab["[PAD]"] == 0  # 确认补齐 token 使用约定编号零

词表： {'[PAD]': 0, '[UNK]': 1, '[BOS]': 2, '[EOS]': 3, '到': 4, '左侧': 5, '托盘': 6, '把': 7, '拿起': 8, '推动': 9, '放到': 10, '方块': 11, '杯子': 12, '红色': 13, '绿色': 14, '蓝色': 15}
第一条指令 id： [2, 8, 13, 12, 3]


## 2. padding 与 mask

batch 必须是规则矩形。短序列补 `[PAD]`，同时用 mask 标出哪些位置来自真实指令。

In [3]:
max_length = max(len(sequence) for sequence in encoded)  # 找到当前 batch 的最长序列
token_ids = torch.full((len(encoded), max_length), vocab["[PAD]"], dtype=torch.long)  # 创建全为 PAD 的整数张量
mask = torch.zeros((len(encoded), max_length), dtype=torch.float32)  # 创建全为零的有效位遮罩
for row, sequence in enumerate(encoded):  # 逐条填入变长序列
    token_ids[row, :len(sequence)] = torch.tensor(sequence)  # 把真实 token 写到每一行前部
    mask[row, :len(sequence)] = 1.0  # 将真实 token 对应位置标为一
print("token_ids shape:", tuple(token_ids.shape))  # 检查 token batch 形状
print(token_ids)  # 显示补齐后的 token id
print("mask:\n", mask)  # 显示与 token 对齐的有效位遮罩
assert token_ids.shape == mask.shape  # 验证 token 与 mask 的时间维完全一致

token_ids shape: (3, 7)
tensor([[ 2,  8, 13, 12,  3,  0,  0],
        [ 2,  7, 15, 11, 10,  6,  3],
        [ 2,  9, 14, 12,  4,  5,  3]])
mask:
 tensor([[1., 1., 1., 1., 1., 0., 0.],
        [1., 1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1., 1.]])


## 3. 比较普通平均与 masked pooling

我们故意让 PAD embedding 不是零向量。这样可以直接看到：若平均时忘记 mask，短句会受到更多补齐位置影响。

In [4]:
embedding = torch.nn.Embedding(len(vocab), 8)  # 创建八维可学习 token 查找表
token_features = embedding(token_ids)  # 把 [B,L] 的 id 查表为 [B,L,D] 向量
plain_mean = token_features.mean(dim=1)  # 错误示范：把 PAD 也纳入普通平均
expanded_mask = mask.unsqueeze(-1)  # 把 mask 扩展为 [B,L,1] 以便逐 token 相乘
masked_sum = (token_features * expanded_mask).sum(dim=1)  # 只累计真实 token 的向量
valid_count = expanded_mask.sum(dim=1).clamp_min(1.0)  # 统计每条指令的有效 token 数并防止除零
masked_mean = masked_sum / valid_count  # 得到不受补齐位置污染的句子向量
difference = torch.linalg.vector_norm(plain_mean - masked_mean, dim=1)  # 计算两种池化结果的距离
print("token features shape:", tuple(token_features.shape))  # 检查 token 表示形状
print("每条指令的池化差异：", difference.detach().numpy().round(4))  # 显示不同长度指令受到的影响
assert masked_mean.shape == (len(commands), 8)  # 验证任务向量形状为 [B,D]
assert difference[0] > difference[1]  # 确认较短指令因 PAD 更多而受到更大影响

token features shape: (3, 7, 8)
每条指令的池化差异： [0.5827 0.     0.    ]


In [5]:
new_command = "拿起 紫色 杯子"  # 构造包含词表外颜色的新指令
new_ids = encode(new_command)  # 使用训练词表编码未见指令
decoded = [inverse_vocab[index] for index in new_ids]  # 把编号还原成可读 token 以检查未知词
print("原指令：", new_command)  # 显示原始未见指令
print("编码后：", decoded)  # 显示紫色被映射为未知 token
assert "[UNK]" in decoded  # 验证词表外词确实触发未知标记

原指令： 拿起 紫色 杯子
编码后： ['[BOS]', '拿起', '[UNK]', '杯子', '[EOS]']


## 结论与练习

token id 只是查表地址；padding 只用于组成 batch；mask 决定哪些位置参与编码和池化。尝试新增更短的指令，预测普通平均与 masked pooling 的差异；再把词级切分改成字符级切分，比较序列长度和未知词数量。